# Tratamento de Lista Geral de Leads

### Objetivo: Esse pipeline tem como objetivo separar a lista de Leads disponível para disparos e consumo

In [ ]:
# Importações básicas
import pandas as pd
import numpy as np
import sys
from pathlib import Path
import os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'src'))

# Adiciona src ao path
sys.path.append('../src')

#

# Utilitários de dados
from data_utils import (
    load_raw_data,
    save_processed_data,
    remove_duplicates,
    handle_missing_values,
    detect_outliers,
    normalize_column,
    process_phone_string,
    process_phone_number,
    clean_and_lower_column,
    flatten_list_to_df,
    remove_buyers_from_dataframe,
    remover_leads_do_dataframe, 
)

CRONOGRAMA_SUBDOMAIN = 'cronogramadosfluentes-xwamel'

# Utilitários SQL
from sql_utils import DatabaseConnection as Dbc, load_query_from_file

# Utilitários de visualização
import matplotlib.pyplot as plt
import seaborn as sns

# Utilitários de API
from api_utils import (
    make_request,
    get_json,
    post_json,
    paginated_request,
    response_to_dataframe
)

# utilitários hotmart
from hotmart_utils import Hotmart, remove_empty_objects

# utilitários tmb
from tmb_utils import TMB   

# Configurações pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# Load Database Driver
db = Dbc()

# Inicializar API Hotmart
hotmart = Hotmart()

# Inicializar API TMB
tmb = TMB()

print('✓ Importações concluídas com sucesso!')

## Load Leads Dataframe

In [ ]:
_df = db.execute_query("SELECT * FROM lead_tracking_prod.conversions c WHERE c.conversion_type_id = '2'")

df = flatten_list_to_df(_df.to_dict(orient='records'))

In [ ]:
tokens = df['conversion_raw_info_token'].unique()
submission_ids = df['conversion_raw_info_submission_id'].unique()

In [ ]:
raw_surveys_responses = load_raw_data("tally_sub.csv")

surveys_to_add = raw_surveys_responses[(~raw_surveys_responses['Submission ID'].isin(submission_ids) & ~raw_surveys_responses['Submission ID'].isin(tokens))]

In [ ]:
"""
Cell generated by Data Wrangler.
"""
def clean_data(surveys_to_add):
    # Filter rows based on columns: 'phone', 'email'
    surveys_to_add = surveys_to_add[(surveys_to_add['phone'].notna()) & (surveys_to_add['email'].notna())]
    return surveys_to_add

surveys_to_add = clean_data(surveys_to_add.copy())
surveys_to_add.head()

In [ ]:
import math

def is_nan(val):
    try:
        import pandas as pd
        return pd.isna(val)
    except ImportError:
        return val != val  # best-effort generic

def clean_nans(d):
    """Recursively remove any keys whose value is nan."""
    if isinstance(d, dict):
        return {k: clean_nans(v) for k, v in d.items() if not is_nan(v)}
    elif isinstance(d, list):
        return [clean_nans(x) for x in d if not is_nan(x)]
    return d

def transform_into_conversion(row):
    output_conversion = {
        "lead_data": {
            "name": row.get("name"),
            "email": {"value": None},
        },
        "conversion_data": {
            "campaign_id": "isca-aulassemanais-s",
            "conversion_type_id": "2",
            "conversion_date": row.get("Submitted at"),
            "utm_source": row.get("utm_source"),
            "utm_medium": row.get("utm_medium"),
            "utm_campaign": row.get("utm_campaign"),
            "utm_content": row.get("utm_content"),
            "utm_term": row.get("utm_term")    
        }
    }

    if row.get("email") and not is_nan(row.get("email")):
        output_conversion["lead_data"]["email"]["value"] = row.get("email")
    
    if row.get("phone") and not is_nan(row.get("phone")):
        output_conversion["lead_data"]["phone"] = process_phone_number(row.get("phone"))
        output_conversion["lead_data"]["phone"]["raw_phone"] = output_conversion["lead_data"]["phone"]["raw_phone_input"]

    conversion_raw_info = {
        "submission_id": row.get("Submission ID"),
        "gender": row.get("gender"),
        "country": row.get("country"),
        "age_range": row.get("age_range"),
        "current_occupation": row.get("current_occupation"),
        "monthy_income": row.get("monthy_income"),
        "biggest_fluency_desire": row.get("biggest_fluency_desire"),
        "english_level": row.get("english_level"),
        "took_english_course": row.get("took_english_course"),
    }

    output_conversion['conversion_data']['conversion_raw_info'] = conversion_raw_info

    return remove_empty_objects(clean_nans(output_conversion))

# Transforma o dataframe surveys_to_add em uma lista de objetos usando a função transform_into_conversion
conversions_list = [
    transform_into_conversion(row._asdict() if hasattr(row, "_asdict") else row.to_dict())
    for _, row in surveys_to_add.iterrows()
]


In [ ]:
inspect_list = flatten_list_to_df(conversions_list)

In [ ]:
import requests
import time
import json
import os

IMPOT_DATAFRAME = conversions_list
BASE_URL = "https://southamerica-east1-aloud-etl.cloudfunctions.net/identity-resolution-http"

responses = []

PERSISTENCE_FILE = "imported_surveys.json"

# Função auxiliar para obter valor de chave profunda (key_path)
def get_deep_key(d, key_path, default=""):
    """Busca valor em d pelo key_path em formato dot (ex: 'conversion_data.conversion_raw_info.transaction')"""
    try:
        for key in key_path.split("."):
            d = d[key]
        return d
    except (KeyError, TypeError):
        return default

# Permite selecionar qual chave (via path) será usada para controle de importação
IMPORT_CONTROL_KEY_PATH = "conversion_data.conversion_raw_info.submission_id"

# Carregue os registros persistidos previamente já enviados, se existir
if os.path.exists(PERSISTENCE_FILE):
    with open(PERSISTENCE_FILE, "r", encoding="utf-8") as f:
        imported_keys = set(json.load(f))
else:
    imported_keys = set()

total_to_import = len(IMPOT_DATAFRAME)
print(f"Total de registros para importar: {total_to_import}")

for idx, item in enumerate(IMPOT_DATAFRAME, start=1):
    import_key = str(get_deep_key(item, IMPORT_CONTROL_KEY_PATH, "")).strip()
    if not import_key:
        print(f"Registro {idx} ignorado: chave de controle '{IMPORT_CONTROL_KEY_PATH}' ausente ou inválida.")
        continue
    if import_key in imported_keys:
        print(f"Registro {idx} já foi importado anteriormente para a chave '{import_key}', ignorando.")
        continue
    
    print(f"Importando registro {idx} de {total_to_import}... (key: {import_key})")
    response = requests.post(BASE_URL, json=item)
    try:
        resp_json = response.json()
    except Exception:
        resp_json = response.text
    responses.append(resp_json)
    
    # Atualize a persistência local imediatamente
    imported_keys.add(import_key)
    with open(PERSISTENCE_FILE, "w", encoding="utf-8") as f:
        json.dump(sorted(list(imported_keys)), f, ensure_ascii=False, indent=2)